# 4) Exploratory Data Analysis (EDA)

This notebook performs Exploratory Data Analysis on the healthcare disease prediction dataset.

The analysis focuses on disease distribution, symptom frequency, missing values, incomplete records, and disease-symptom relationships.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import csv
from io import StringIO

In [ ]:
# Load dataset from GitHub

url = "https://raw.githubusercontent.com/vyasanbmathew2008/Team-6/main/healthcare-disease-prediction/dataset/healthcare_dataset.csv"

response = requests.get(url)
response.raise_for_status()

reader = csv.reader(StringIO(response.text))
rows = list(reader)

header = rows[0]
data_rows = rows[1:]

max_fields = max(len(row) for row in data_rows)

while len(header) < max_fields:
    header.append(f"Symptom_{len(header)}")

fixed_rows = []

for row in data_rows:
    if len(row) < max_fields:
        row = row + [""] * (max_fields - len(row))
    elif len(row) > max_fields:
        row = row[:max_fields]
    fixed_rows.append(row)

df = pd.DataFrame(fixed_rows, columns=header)

print("Dataset loaded successfully!")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

In [ ]:
# Display first five records
df.head()

In [ ]:
# Display dataset information
df.info()

In [ ]:
# Display basic statistics
df.describe(include="all").T

## 1. Identify Disease and Symptom Columns

In [ ]:
disease_column = "Disease"

symptom_columns = [
    column for column in df.columns
    if column.startswith("Symptom_")
]

print("Disease column:", disease_column)
print("Number of symptom columns:", len(symptom_columns))
print("Symptom columns:", symptom_columns)

## 2. Missing Value Analysis

In [ ]:
# Convert empty strings to NaN for analysis
eda_df = df.copy()
eda_df = eda_df.replace(r"^\s*$", np.nan, regex=True)

missing_values = eda_df.isnull().sum()

missing_summary = pd.DataFrame({
    "Missing Values": missing_values,
    "Percentage": (missing_values / len(eda_df) * 100).round(2)
})

missing_summary

In [ ]:
# Plot missing values
missing_plot = missing_summary[missing_summary["Missing Values"] > 0]

if len(missing_plot) > 0:
    plt.figure(figsize=(12, 5))
    sns.barplot(
        x=missing_plot.index,
        y=missing_plot["Missing Values"]
    )
    plt.title("Missing Values by Column")
    plt.xlabel("Column")
    plt.ylabel("Missing Values")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
else:
    print("No missing values found.")

## 3. Disease Distribution

In [ ]:
disease_counts = df[disease_column].value_counts()

print("Number of unique diseases:", df[disease_column].nunique())
disease_counts

In [ ]:
# Disease distribution percentages
disease_distribution = pd.DataFrame({
    "Count": disease_counts,
    "Percentage": (disease_counts / len(df) * 100).round(2)
})

disease_distribution

In [ ]:
# Plot disease distribution
plt.figure(figsize=(12, 8))

sns.countplot(
    data=df,
    y=disease_column,
    order=disease_counts.index
)

plt.title("Distribution of Diseases")
plt.xlabel("Number of Records")
plt.ylabel("Disease")
plt.tight_layout()
plt.show()

## 4. Symptom Frequency Analysis

In [ ]:
# Combine all symptom columns
all_symptoms = pd.concat(
    [df[column] for column in symptom_columns],
    ignore_index=True
)

# Remove empty values
all_symptoms = all_symptoms[
    all_symptoms.notna() &
    (all_symptoms.astype(str).str.strip() != "")
]

symptom_counts = all_symptoms.value_counts()

print("Total symptom occurrences:", len(all_symptoms))
print("Unique symptoms:", all_symptoms.nunique())

In [ ]:
# Display symptom frequencies
symptom_frequency = symptom_counts.reset_index()
symptom_frequency.columns = ["Symptom", "Frequency"]

symptom_frequency.head(30)

In [ ]:
# Plot top 20 symptoms
top_symptoms = symptom_frequency.head(20)

plt.figure(figsize=(10, 7))
sns.barplot(
    data=top_symptoms,
    y="Symptom",
    x="Frequency"
)

plt.title("Top 20 Most Frequent Symptoms")
plt.xlabel("Frequency")
plt.ylabel("Symptom")
plt.tight_layout()
plt.show()

## 5. Number of Symptoms per Record

In [ ]:
# Count available symptoms in each record
symptom_count = (
    df[symptom_columns]
    .replace(r"^\s*$", np.nan, regex=True)
    .notna()
    .sum(axis=1)
)

print("Minimum symptoms in a record:", symptom_count.min())
print("Maximum symptoms in a record:", symptom_count.max())
print("Average symptoms per record:", round(symptom_count.mean(), 2))

In [ ]:
# Plot symptom count distribution
plt.figure(figsize=(9, 5))
sns.countplot(
    x=symptom_count
)

plt.title("Number of Symptoms per Record")
plt.xlabel("Number of Symptoms")
plt.ylabel("Number of Records")
plt.tight_layout()
plt.show()

## 6. Disease vs Number of Symptoms

In [ ]:
# Calculate average symptoms for each disease
disease_symptom_analysis = pd.DataFrame({
    "Disease": df[disease_column],
    "Symptom_Count": symptom_count
})

disease_symptom_summary = (
    disease_symptom_analysis
    .groupby("Disease")["Symptom_Count"]
    .agg(["count", "mean", "min", "max"])
    .sort_values("mean", ascending=False)
)

disease_symptom_summary

## 7. Disease-Symptom Relationship

In [ ]:
# Create disease-symptom frequency matrix
top_15_symptoms = symptom_counts.head(15).index.tolist()
diseases = df[disease_column].unique()

matrix = pd.DataFrame(
    0,
    index=diseases,
    columns=top_15_symptoms
)

for disease in diseases:
    disease_rows = df[df[disease_column] == disease]
    
    for symptom in top_15_symptoms:
        matrix.loc[disease, symptom] = (
            disease_rows[symptom_columns]
            .isin([symptom])
            .any(axis=1)
            .sum()
        )

matrix.head()

In [ ]:
# Plot disease-symptom heatmap
plt.figure(figsize=(14, 10))

sns.heatmap(
        matrix,
        cmap="YlGnBu",
        linewidths=0.2
)

plt.title("Disease-Symptom Frequency Heatmap")
plt.xlabel("Symptoms")
plt.ylabel("Disease")
plt.tight_layout()
plt.show()

## 8. Categorical Feature Analysis

In [ ]:
categorical_columns = df.select_dtypes(include="object").columns.tolist()

print("Categorical/Text Columns:")

for column in categorical_columns:
    print(column)

print("\nTotal categorical columns:", len(categorical_columns))

In [ ]:
# Unique value count for categorical columns
categorical_summary = pd.DataFrame({
    "Column": categorical_columns,
    "Unique Values": [
        df[column].replace("", np.nan).nunique(dropna=True)
        for column in categorical_columns
    ]
})

categorical_summary

## 9. Incomplete Records

In [ ]:
max_symptoms = len(symptom_columns)

incomplete_records = df[symptom_count < max_symptoms]
complete_records = df[symptom_count == max_symptoms]

print("Total records:", len(df))
print("Complete records:", len(complete_records))
print("Incomplete records:", len(incomplete_records))
print("Incomplete percentage:", round(len(incomplete_records) / len(df) * 100, 2), "%")

In [ ]:
# Display sample incomplete records
incomplete_records.head(10)

## 10. Duplicate Analysis

In [ ]:
duplicate_count = df.duplicated().sum()
duplicate_percentage = duplicate_count / len(df) * 100

print("Duplicate records:", duplicate_count)
print("Duplicate percentage:", round(duplicate_percentage, 2), "%")

## 11. EDA Summary

In [ ]:
print("=" * 60)
print("EXPLORATORY DATA ANALYSIS SUMMARY")
print("=" * 60)
print("Total records:", len(df))
print("Total columns:", len(df.columns))
print("Number of diseases:", df[disease_column].nunique())
print("Number of symptom columns:", len(symptom_columns))
print("Number of unique symptoms:", all_symptoms.nunique())
print("Complete records:", len(complete_records))
print("Incomplete records:", len(incomplete_records))
print("Duplicate records:", duplicate_count)
print("Average symptoms per record:", round(symptom_count.mean(), 2))
print("=" * 60)

# Conclusion

Exploratory Data Analysis was performed to understand the structure and distribution of the healthcare disease prediction dataset. Disease frequencies, symptom frequencies, missing values, incomplete records, duplicate records, and disease-symptom relationships were analyzed.

The findings from this analysis will help guide the data preprocessing and feature engineering stages before machine learning model training.